In [1]:
# PLANT DISEASE CNN TRAINING

import os
import pandas as pd
import json
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Settings

DATASET_PATH = "Dataset"

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 15

# Data Preprocessing


datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Training data
train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

# Validation data
validation_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

# 3. Class Names

class_names = list(train_data.class_indices.keys())

print("\nClasses:")
print(class_names)

print("\nNumber of Classes:")
print(train_data.num_classes)

#  CNN Model

model = Sequential([

    Input(shape=(128, 128, 3)),

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.5),

    Dense(train_data.num_classes, activation="softmax")
])

# Compile Model

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Early Stopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# Train Model

history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=10,
    callbacks=[early_stopping]
)

#  Create Model Folder

os.makedirs("model", exist_ok=True)

# Save Model

model.save("model/plant_disease_model.keras")

# Save Class Names

with open("model/class_names.json", "w") as file:
    json.dump(class_names, file)

print("\n================================")
print("MODEL TRAINING COMPLETED!")
print("================================")

print("Model saved at:")
print("model/plant_disease_model.keras")

print("\nClass names saved at:")
print("model/class_names.json")

Found 8931 images belonging to 10 classes.
Found 2229 images belonging to 10 classes.

Classes:
['Corn_(maize)___Common_rust_', 'Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold']

Number of Classes:
10
Epoch 1/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 328s 1s/step - accuracy: 0.4978 - loss: 1.4472 - val_accuracy: 0.6788 - val_loss: 1.0360
Epoch 2/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 351s 1s/step - accuracy: 0.6924 - loss: 0.8835 - val_accuracy: 0.7129 - val_loss: 1.0618
Epoch 3/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 470s 2s/step - accuracy: 0.7562 - loss: 0.7001 - val_accuracy: 0.7636 - val_loss: 1.0170
Epoch 4/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 538s 2s/step - accuracy: 0.7914 - loss: 0.5839 - val_accuracy: 0.7743 - val_loss: 0.9984
Epoch 5/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 521s 2s/step - accuracy: 0.8172 - loss: 0.5119 - val_accuracy: 0.7932 - val

In [2]:
import numpy as np
from tensorflow.keras.preprocessing import image

# Test image ka path
img_path = r"C:\Users\hp\Desktop\Agro AI\test_images\early.jfif"

# Image load karo
img = image.load_img(
    img_path,
    target_size=(128, 128)
)

# Image ko array mein convert karo
img_array = image.img_to_array(img)

# Normalize
img_array = img_array / 255.0

# Batch dimension add
img_array = np.expand_dims(
    img_array,
    axis=0
)

# Prediction
prediction = model.predict(img_array)

# Predicted class index
predicted_index = np.argmax(prediction)

# Class names automatically get karo
class_names = list(train_data.class_indices.keys())

# Result
predicted_class = class_names[predicted_index]
confidence = np.max(prediction) * 100

print("Predicted Disease:", predicted_class)
print(f"Confidence: {confidence:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Predicted Disease: Corn_(maize)___Common_rust_
Confidence: 68.72%
